# Phase 5 - Latent-Adaptive Best-of-N

Esta fase pergunta se um orcamento adaptativo pode manter acuracia enquanto reduz tentativas, tokens e tempo. A politica pode usar score latente da Fase 3 quando houver direcoes disponiveis.

Configuramos caminhos e importamos bibliotecas. O notebook permanece valido mesmo sem resultados.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
PHASE5_DIR = ROOT / 'runs' / 'phase5'
PHASE5_DIR

Carregamos summaries das politicas adaptativas e transformamos cada run em uma linha comparavel.

In [ ]:
rows = []
for path in sorted(PHASE5_DIR.rglob('*_summary.json')) if PHASE5_DIR.exists() else []:
    data = json.loads(path.read_text(encoding='utf-8'))
    policy = data.get('policy', {})
    rows.append({
        'run': path.stem.replace('_summary', ''),
        'policy': policy.get('name'),
        'max_n': policy.get('max_n'),
        'accuracy': data.get('observed_best_of_n') or data.get('success_rate'),
        'mean_attempts': data.get('mean_attempts_until_success_or_budget') or data.get('mean_attempts'),
        'mean_tokens': data.get('mean_tokens_until_success_or_budget') or data.get('mean_tokens'),
        'budget_saved': data.get('budget_saved_vs_fixed_n'),
        'tokens_per_success': data.get('tokens_per_success'),
    })

phase5 = pd.DataFrame(rows)
phase5

A fronteira abaixo revela politicas dominantes: melhor acuracia no eixo vertical e menor custo no eixo horizontal.

In [ ]:
if phase5.empty:
    print('Sem resultados da Fase 5 ainda.')
else:
    ax = phase5.plot.scatter(x='mean_tokens', y='accuracy', s=80)
    for _, row in phase5.iterrows():
        ax.annotate(str(row['policy']), (row['mean_tokens'], row['accuracy']))
    ax.set_title('Politicas adaptativas: custo vs acuracia')
    plt.show()

Esta tabela ordena as politicas por economia de orcamento e permite discutir trade-offs em termos de tokens por sucesso.

In [ ]:
if phase5.empty:
    ranking = pd.DataFrame()
else:
    ranking = phase5.sort_values(['accuracy', 'budget_saved'], ascending=[False, False])
ranking

Comando minimo para gerar dados desta fase:

`python scripts/run_adaptive_bon.py --limit 5 --max-n 3 --policy latent_adaptive`